# ⚡ MetadataPro — FLUX.1 Image Generator (Lightning.ai Edition)

**নির্দেশনা (Instructions):**
1. Lightning.ai Studio-তে একটি নতুন Studio তৈরি করুন (PyTorch বা Default টেমপ্লেট দিয়ে)।
2. ডানদিকের প্যানেল থেকে **GPU (L4, A10G বা 4090)** সিলেক্ট করে স্টার্ট করুন।
3. এই নোটবুকটি ওপেন করে নিচের কোড সেলটির **Run** (▶️) বাটনে ক্লিক করুন।
4. রান হয়ে গেলে নিচে একটি **লাইভ লিংক** (`trycloudflare.com`) আসবে।
5. আপনার ডেস্কটপ অ্যাপটি স্বয়ংক্রিয়ভাবে সেই লিংকের সাথে কানেক্ট হয়ে যাবে। যদি না হয়, লিংকটি কপি করে অ্যাপে ম্যানুয়ালি পেস্ট করুন।

In [ ]:
# 🚀 MetadataPro — FLUX.1 Setup for Lightning.ai
import os
import re
import socket
import subprocess
import threading
import time
import torch
from IPython.display import display, HTML

BASE_DIR = os.getcwd()
COMFY_DIR = os.path.join(BASE_DIR, 'ComfyUI')

print("=" * 60)
print("⚡ Lightning.ai: MetadataPro — FLUX.1 Setup & Run")
print("=" * 60)

# 1. GPU Check
if not torch.cuda.is_available():
  print("⚠️ WARNING: GPU IS NOT ENABLED!")
  gpu_name = "CPU"
  gpu_mem_gb = 0
else:
  gpu_name = torch.cuda.get_device_name(0)
  gpu_mem_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
  print(f"✅ GPU Detected: {gpu_name} ({gpu_mem_gb} GB VRAM)")

# Define models based on GPU VRAM
if gpu_mem_gb >= 20:
  SCHNELL_FILE = "flux1-schnell-Q8_0.gguf"
  SCHNELL_URL = "https://huggingface.co/city96/FLUX.1-schnell-gguf/resolve/main/flux1-schnell-Q8_0.gguf"
  DEV_FILE = "flux1-dev-Q8_0.gguf"
  DEV_URL = "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q8_0.gguf"
  QUALITY_TAG = "Q8_0 (High Quality)"
elif gpu_mem_gb >= 8:
  SCHNELL_FILE = "flux1-schnell-Q5_K_M.gguf"
  SCHNELL_URL = "https://huggingface.co/city96/FLUX.1-schnell-gguf/resolve/main/flux1-schnell-Q5_K_M.gguf"
  DEV_FILE = "flux1-dev-Q5_K_M.gguf"
  DEV_URL = "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q5_K_M.gguf"
  QUALITY_TAG = "Q5_K_M (Balanced)"
else:
  SCHNELL_FILE = "flux1-schnell-Q4_K_S.gguf"
  SCHNELL_URL = "https://huggingface.co/city96/FLUX.1-schnell-gguf/resolve/main/flux1-schnell-Q4_K_S.gguf"
  DEV_FILE = "flux1-schnell-Q4_K_S.gguf"
  DEV_URL = SCHNELL_URL
  QUALITY_TAG = "Q4_K_S (Speed/Low VRAM/CPU)"

print(f"🎯 Target Quality Level: {QUALITY_TAG}")
print("=" * 60)

# 2. Helper Functions
def run_cmd(cmd, cwd=None):
  try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy:
      if cwd:
        old_cwd = os.getcwd()
        os.chdir(cwd)
      ipy.system(cmd)
      if cwd:
        os.chdir(old_cwd)
      return
  except Exception:
    pass
  shell = isinstance(cmd, str)
  subprocess.run(cmd, shell=shell, check=True, cwd=cwd)

def download_file(url, dest_path):
  if os.path.exists(dest_path) and os.path.getsize(dest_path) > 100_000_000:
    print(f"ℹ️ {os.path.basename(dest_path)} is already cached. Skipping download.")
    return
  print(f"📥 Downloading {os.path.basename(dest_path)}...")
  run_cmd(f'wget -c -q --show-progress "{url}" -O "{dest_path}"')

# 3. Clone & Install ComfyUI
if not os.path.exists(os.path.join(COMFY_DIR, 'main.py')):
  print("🚀 Cloning ComfyUI repository...")
  run_cmd(f'git clone https://github.com/comfyanonymous/ComfyUI "{COMFY_DIR}"')
else:
  print("ℹ️ ComfyUI already cloned.")

print("📦 Installing python dependencies...")
run_cmd('pip install -q -r requirements.txt', cwd=COMFY_DIR)
run_cmd('pip install -q gguf')

# Install GGUF Custom Nodes
gguf_node_dir = os.path.join(COMFY_DIR, 'custom_nodes', 'ComfyUI-GGUF')
if not os.path.exists(os.path.join(gguf_node_dir, '__init__.py')):
  print("🚀 Installing ComfyUI-GGUF custom nodes...")
  run_cmd(f'git clone https://github.com/city96/ComfyUI-GGUF "{gguf_node_dir}"')

# 4. Setup Directories
unet_dir = os.path.join(COMFY_DIR, 'models', 'unet')
clip_dir = os.path.join(COMFY_DIR, 'models', 'clip')
vae_dir = os.path.join(COMFY_DIR, 'models', 'vae')
os.makedirs(unet_dir, exist_ok=True)
os.makedirs(clip_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)

# 5. Download Models
print("\n" + "=" * 60)
print("📥 Downloading FLUX.1 Models (This may take a few minutes)")
print("=" * 60)
download_file(SCHNELL_URL, os.path.join(unet_dir, SCHNELL_FILE))
if DEV_FILE != SCHNELL_FILE:
  download_file(DEV_URL, os.path.join(unet_dir, DEV_FILE))

download_file('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
              os.path.join(clip_dir, 't5xxl_fp8_e4m3fn.safetensors'))
download_file('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
              os.path.join(clip_dir, 'clip_l.safetensors'))
download_file('https://huggingface.co/camenduru/FLUX.1-dev-ungated/resolve/main/ae.safetensors',
              os.path.join(vae_dir, 'ae.safetensors'))

# 6. Create Symlinks
all_expected_schnell = ["flux1-schnell-Q8_0.gguf", "flux1-schnell-Q5_K_M.gguf", "flux1-schnell-Q4_K_S.gguf"]
all_expected_dev = ["flux1-dev-Q8_0.gguf", "flux1-dev-Q5_K_M.gguf", "flux1-schnell-Q4_K_S.gguf"]

def make_symlink(target_name, link_name, directory):
  target_path = os.path.join(directory, target_name)
  link_path = os.path.join(directory, link_name)
  if os.path.exists(target_path):
    if os.path.exists(link_path) or os.path.islink(link_path):
      if os.path.basename(os.path.realpath(link_path)) == target_name: return
      try: os.remove(link_path)
      except Exception: pass
    if not os.path.exists(link_path):
      try: os.symlink(target_name, link_path)
      except Exception: pass

for link_name in all_expected_schnell:
  if link_name != SCHNELL_FILE: make_symlink(SCHNELL_FILE, link_name, unet_dir)
for link_name in all_expected_dev:
  if link_name != DEV_FILE: make_symlink(DEV_FILE, link_name, unet_dir)

# 7. Download and Start Cloudflare Tunnel
cloudflared_path = os.path.join(BASE_DIR, 'cloudflared')
if not os.path.exists(cloudflared_path):
  print("\n📥 Downloading Cloudflare tunnel...")
  run_cmd(f'wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O "{cloudflared_path}"')
  run_cmd(f'chmod +x "{cloudflared_path}"')

def wait_and_tunnel(port):
  print("⏳ Waiting for ComfyUI server to boot...")
  for _ in range(180):
    time.sleep(1)
    s = socket.socket()
    try:
      if s.connect_ex(('127.0.0.1', port)) == 0: break
    finally:
      s.close()
  else:
    print("❌ ERROR: ComfyUI server failed to start in 3 minutes.")
    return

  print("🎉 ComfyUI is up! Initializing Tunnel...")
  p = subprocess.Popen([cloudflared_path, 'tunnel', '--url', f'http://127.0.0.1:{port}'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
  for line in p.stdout:
    line_str = line.decode('utf-8', errors='ignore')
    m = re.search(r'https://[-a-zA-Z0-9]+[.]trycloudflare[.]com', line_str)
    if m:
      url = m.group(0)
      print("\n" + "🔥" * 25)
      print("🚀 SUCCESS! COPY THIS LINK TO YOUR DESKTOP APP:")
      print(f"\n   👉 {url} 👈\n")
      display(HTML(f'<a href="{url}" target="_blank" style="font-weight:bold;color:#2563eb;font-size:16px;">{url}</a>'))
      break

threading.Thread(target=wait_and_tunnel, args=(8188,), daemon=True).start()

# 8. Run ComfyUI Server
print("🚀 Starting ComfyUI server...")
os.chdir(COMFY_DIR)
cmd = 'python main.py --dont-print-server --enable-cors-header "*"'
run_cmd(cmd)